In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.signal import argrelextrema
from IPython.display import display
import os

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Load all data files
dataframes = {}
for i in os.listdir('Assets'):
    dataframes[i] = pd.read_csv(os.path.join('Assets', i))
    print(f'✅ Loaded {i}')

# Merge sales transaction with product master
sale_data = pd.merge(
    pd.read_csv('Assets/sales_transaction.csv', parse_dates=['datetime']),
    dataframes['product_master.csv'],
    on='product_id',
    how='inner'
)

# Rename columns to English
sale_data.rename(columns={
    'sales_transaction_id': 'transaction_id',
    'product_taxonomies': 'category'
}, inplace=True)

print(f'\n✅ Data merged successfully')
print(f'Shape: {sale_data.shape}')
print(f'Columns: {list(sale_data.columns)}')

In [ ]:
# Define time period function
def get_time_period(hour):
    if 7 <= hour <= 11:
        return 'Morning'
    elif 12 <= hour <= 15:
        return 'Afternoon'
    else:
        return 'Evening'

# Add hour and time_period
sale_data['hour'] = sale_data['datetime'].dt.hour
sale_data['time_period'] = sale_data['hour'].apply(get_time_period)

print('✅ Time period classification added')
print(f'Unique periods: {sorted(sale_data["time_period"].unique())}')
print(f'\nSample data:')
display(sale_data[['transaction_id', 'product_name', 'category', 'qty', 'hour', 'time_period']].head(10))

In [ ]:
# Create hourly sales dataframe
hourly_sales = sale_data[['product_name', 'hour', 'qty']].groupby(['product_name', 'hour'])['qty'].sum().reset_index()
hourly_sales['time_period'] = hourly_sales['hour'].apply(get_time_period)

# Summary statistics
print('✅ Hourly sales dataframe created')
print(f'Shape: {hourly_sales.shape}')
print(f'\nSummary by time period:')
summary = hourly_sales.groupby('time_period')['qty'].agg(['sum', 'count', 'mean']).round(2)
summary.columns = ['Total Sales', 'Records', 'Avg per Record']
display(summary)

In [ ]:
# Define color palette
category_colors = {
    'Beverage': 'limegreen',
    'Food': 'steelblue',
    'Bakery': 'orange'
}

# Prepare data
plot_data = sale_data.groupby(['product_name', 'category'])['qty'].sum().sort_values(ascending=False).reset_index()

# Create barplot
plt.figure(figsize=(14, 8))
sns.barplot(x='qty', y='product_name', hue='category', data=plot_data, palette=category_colors)
plt.xlabel('Total Sales Quantity', fontsize=12, fontweight='bold')
plt.ylabel('Product Name', fontsize=12, fontweight='bold')
plt.title('Total Sales by Product and Category', fontsize=14, fontweight='bold')
plt.legend(title='Category', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()

In [ ]:
# Sales by category only
category_data = sale_data.groupby('category')['qty'].sum().sort_values(ascending=False).reset_index()

# Create barplot
plt.figure(figsize=(10, 6))
sns.barplot(x='qty', y='category', data=category_data, hue='category', palette=category_colors, legend=False)
plt.xlabel('Total Sales Quantity', fontsize=12, fontweight='bold')
plt.ylabel('Category', fontsize=12, fontweight='bold')
plt.title('Total Sales by Category', fontsize=14, fontweight='bold')
plt.grid(axis='x', alpha=0.3)

# Add values on bars
for i, v in enumerate(category_data['qty']):
    plt.text(v + 2, i, str(int(v)), va='center', fontweight='bold')
    
plt.tight_layout()

In [ ]:
# Define time period colors
time_period_colors = {
    'Morning': 'lightcoral',
    'Afternoon': 'lightyellow',
    'Evening': 'lightblue'
}

time_period_order = ['Morning', 'Afternoon', 'Evening']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Subplot 1: Histogram
for period in time_period_order:
    period_data = hourly_sales[hourly_sales['time_period'] == period]['qty']
    axes[0].hist(period_data, alpha=0.6, label=period, color=time_period_colors[period], bins=8, edgecolor='black')

axes[0].set_xlabel('Sales Quantity per Hour', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Frequency', fontsize=12, fontweight='bold')
axes[0].set_title('Distribution of Hourly Sales by Time Period', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Subplot 2: Total sales by time period
time_period_data = sale_data.groupby('time_period')['qty'].sum().reindex(time_period_order)
colors = [time_period_colors.get(period, 'gray') for period in time_period_order]

axes[1].bar(time_period_data.index, time_period_data.values, color=colors, edgecolor='black', linewidth=1.5)
axes[1].set_xlabel('Time Period', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Total Sales Quantity', fontsize=12, fontweight='bold')
axes[1].set_title('Total Sales by Time Period', fontsize=12, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

# Add values on bars
for i, (period, value) in enumerate(zip(time_period_data.index, time_period_data.values)):
    axes[1].text(i, value + 5, str(int(value)), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()

In [ ]:
# Prepare data for stacked bar chart
pivot_data = sale_data.groupby(['product_name', 'time_period'])['qty'].sum().unstack(fill_value=0)
pivot_data = pivot_data[time_period_order]

# Create stacked bar chart
fig, ax = plt.subplots(figsize=(16, 8))

pivot_data.plot(kind='bar', stacked=True, ax=ax,
                color=[time_period_colors[period] for period in time_period_order],
                edgecolor='black', linewidth=0.5)

ax.set_xlabel('Product Name', fontsize=12, fontweight='bold')
ax.set_ylabel('Sales Quantity', fontsize=12, fontweight='bold')
ax.set_title('Product Sales by Time Period (Stacked)', fontsize=14, fontweight='bold')
ax.legend(title='Time Period', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(axis='y', alpha=0.3)

# Add numbers on segments
for container in ax.containers:
    ax.bar_label(container, label_type='center', fontweight='bold', fontsize=8)

# Add totals on top
totals = pivot_data.sum(axis=1)
for i, total in enumerate(totals):
    ax.text(i, total + 2, str(int(total)), ha='center', va='bottom',
            fontweight='bold', fontsize=10, color='darkred',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))

plt.xticks(rotation=45, ha='right')
plt.tight_layout()

In [ ]:
# Get unique products
products = sorted(hourly_sales['product_name'].unique())
num_products = len(products)
cols = 3
rows = (num_products + cols - 1) // cols

# Create subplots
fig, axes = plt.subplots(rows, cols, figsize=(16, 4*rows))
axes = axes.flatten()

for idx, product in enumerate(products):
    product_data = hourly_sales[hourly_sales['product_name'] == product].sort_values('hour')
    hours = product_data['hour'].values
    quantities = product_data['qty'].values
    
    # Plot actual sales
    axes[idx].plot(hours, quantities, marker='o', linewidth=2, markersize=6, color='steelblue', label='Sales')
    
    # Add trend line
    z = np.polyfit(hours, quantities, 1)
    p = np.poly1d(z)
    trend_line = p(hours)
    axes[idx].plot(hours, trend_line, linestyle='--', linewidth=2, color='red', label='Trend')
    
    # Find changepoints
    local_max = argrelextrema(quantities, np.greater, order=1)[0]
    local_min = argrelextrema(quantities, np.less, order=1)[0]
    
    if len(local_max) > 0:
        axes[idx].scatter(hours[local_max], quantities[local_max], color='green', s=100, marker='^', label='Peak', zorder=5)
    if len(local_min) > 0:
        axes[idx].scatter(hours[local_min], quantities[local_min], color='orange', s=100, marker='v', label='Valley', zorder=5)
    
    axes[idx].set_xlabel('Hour')
    axes[idx].set_ylabel('Sales Quantity')
    axes[idx].set_title(f'{product}')
    axes[idx].set_xticks(range(7, 21))
    axes[idx].legend(loc='best', fontsize=8)
    axes[idx].grid(True, alpha=0.3)

# Hide unused subplots
for idx in range(num_products, len(axes)):
    axes[idx].axis('off')

plt.tight_layout()

In [ ]:
# Prepare data for scatter plot
scatter_data = sale_data.groupby(['product_name', 'time_period'])['qty'].sum().reset_index()

# Create scatter plot
plt.figure(figsize=(16, 8))
sns.scatterplot(
    data=scatter_data,
    x='product_name',
    y='qty',
    hue='time_period',
    hue_order=time_period_order,
    palette=time_period_colors,
    s=300,
    alpha=0.7,
    edgecolor='black',
    linewidth=2
)

plt.xlabel('Product Name', fontsize=12, fontweight='bold')
plt.ylabel('Total Sales Quantity', fontsize=12, fontweight='bold')
plt.title('Total Sales by Product and Time Period (Scatter Plot)', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Time Period', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()